# AMS — os 30 casos do Lab 06

Este notebook gera o `D2/lab06-agente-baseline/casos-dataset.csv`, usado pelo `src/ams/baseline.py`,
e registra por que escolhi estes casos.

In [47]:
import numpy as np
import pandas as pd

from ams.dados import carregar

dataset = carregar()
df = dataset.quadro.copy()
RAIZ = dataset.caminho.parents[2]

df["diferenca_temp_k"] = (df["temperatura_processo"] - 273.15) - (df["temperatura_ar"] - 273.15)
df["potencia_w"] = (df["torque"] * df["rotacao"] * 2 * np.pi / 60).round(0)
# um ciclo novo começa quando o desgaste volta a zero: a troca da ferramenta
df["ciclo"] = (df["desgaste"].diff() < -50).cumsum()

## O AI4I

- São 10.000 leituras em sequência (`udi`), com 339 falhas (3,4%).
- As medidas são temperatura do ar e do processo, rotação, torque e desgaste da ferramenta.
- Os modos de falha são TWF (desgaste), HDF (calor), PWF (potência), OSF (sobrecarga) e RNF (aleatória).
- L, M e H são a qualidade do produto, guardada em `tipo_maquina`.
- Não há máquina, data, vibração, idade nem tempo desde a manutenção. Essas colunas do exemplo do lab
  saíram do conjunto.
- O desgaste sobe e zera 119 vezes, formando 120 ciclos de ferramenta. O `ciclo` é o mais próximo de
  uma identidade de máquina, e a memória do agente usa essa coluna.

## Os casos

Os sinais vêm do `01-eda.ipynb`: as falhas se concentram com desgaste acima de 200 min, e a diferença
de temperatura abaixo de 8,6 K dispara a falha HDF. Temperatura alta não indica falha: os 5% de
registros mais quentes falham menos que a média (1,9% contra 3,4%).

São 6 falhas em 30 casos (20%). É mais que os 3,4% do AI4I, porque 1 falha não deixaria casos difíceis.

| Grupo | Casos | Por que está aqui |
|---|---:|---|
| Fácil: operação normal | 17 | Leituras no meio da faixa, sem falha |
| Fácil: progressão | 3 | A mesma ferramenta (ciclo 114) passa de "pode operar" para "precisa parar" |
| Fácil: diferença baixa | 2 | Falhas HDF que alimentam a memória (ciclo 56) |
| Difícil: desgaste alto sem falha | 3 | Desgaste alto não garante falha |
| Difícil: potência fora da faixa | 1 | Falha que desgaste e diferença de temperatura não mostram |
| Difícil: memória | 2 | Depois de 2 falhas no ciclo 56: um caso com falha e um sem |
| Atípico: RNF | 2 | O único RNF com falha e o RNF sem falha com menor desgaste |

Os ciclos 56 e 114 são sequências reais, escolhidas de propósito. O 56 é o único ciclo com duas falhas
HDF, depois uma falha fora das regras reativas e depois um caso normal, então testa a memória. Por isso,
os 30 casos mostram se o agente funciona, e não quanto ele vale em operação.

RNF, falhas sem modo marcado e o ciclo 56 ficam fora do sorteio dos outros grupos.

In [48]:
SEMENTE = 42
DESGASTE_ALTO = 200  # min
DELTA_HDF = 8.6      # K
MODOS = ["twf", "hdf", "pwf", "osf", "rnf"]
CICLO_DA_MEMORIA = 56

SEQUENCIA_DA_MEMORIA = {
    4738: "fácil: diferença baixa com falha",
    4740: "fácil: diferença baixa com falha",
    4779: "difícil: memória com falha",
    4790: "difícil: memória sem falha",
}
PROGRESSAO = [9622, 9632, 9660]  # ciclo 114

coerente = ~df["rnf"] & (df["falha"] == df[MODOS].any(axis=1)) & (df["ciclo"] != CICLO_DA_MEMORIA)
falhou = coerente & df["falha"]
sem_falha = coerente & ~df["falha"]
desgaste_alto = df["desgaste"] > DESGASTE_ALTO
delta_normal = df["diferenca_temp_k"] >= DELTA_HDF
operacao_normal = (
    df["desgaste"].between(50, 150)
    & df["diferenca_temp_k"].between(9, 11)
    & df["potencia_w"].between(4000, 8500)
)

GRUPOS = {
    "fácil: operação normal sem falha": (sem_falha & operacao_normal & ~df["udi"].isin(PROGRESSAO), 17),
    "difícil: potência fora da faixa": (falhou & ~desgaste_alto & delta_normal & df["pwf"], 1),
    "difícil: desgaste alto sem falha": (sem_falha & desgaste_alto & delta_normal, 3),
}
sorteados = [
    df[filtro].sample(quantidade, random_state=SEMENTE).assign(grupo=nome)
    for nome, (filtro, quantidade) in GRUPOS.items()
]

memoria = df[df["udi"].isin(SEQUENCIA_DA_MEMORIA)].assign(
    grupo=lambda linhas: linhas["udi"].map(SEQUENCIA_DA_MEMORIA))
progressao = df[df["udi"].isin(PROGRESSAO)].assign(grupo="fácil: progressão")
atipicos = pd.concat([
    df[df["rnf"] & df["falha"]],
    df[df["rnf"] & ~df["falha"]].nsmallest(1, "desgaste"),
]).assign(grupo="atípico: RNF")

casos = pd.concat([*sorteados, memoria, progressao, atipicos]).sort_values("udi")
assert casos["udi"].is_unique and len(casos) == 30 and casos["falha"].sum() == 6

## Os rótulos

| Situação | `acao_correta` |
|---|---|
| Falha com modo físico (TWF, HDF, PWF ou OSF) | `parar_para_manutencao` |
| RNF sem falha, ou desgaste acima de 200 min sem falha | `inspecionar_antes_de_operar` |
| Demais casos | `seguir_operando` |

In [49]:
modo_fisico = casos[["twf", "hdf", "pwf", "osf"]].any(axis=1)
sinal_ambiguo = casos["rnf"] | (casos["desgaste"] > DESGASTE_ALTO)

conjunto = pd.DataFrame({
    "id_caso": [f"C{i:02d}" for i in range(1, len(casos) + 1)],
    "udi": casos["udi"].to_numpy(),
    "ciclo": casos["ciclo"].to_numpy(),
    "tipo_maquina": casos["tipo"].astype(str).to_numpy(),
    "desgaste_min": casos["desgaste"].to_numpy(),
    "delta_temperatura_k": casos["diferenca_temp_k"].round(1).to_numpy(),
    "rotacao_rpm": casos["rotacao"].to_numpy(),
    "torque_nm": casos["torque"].to_numpy(),
    "acao_correta": np.select(
        [casos["falha"] & modo_fisico, ~casos["falha"] & sinal_ambiguo],
        ["parar_para_manutencao", "inspecionar_antes_de_operar"], default="seguir_operando"),
    "falhou_depois": casos["falha"].astype(int).to_numpy(),
    "grupo": casos["grupo"].to_numpy(),
})

caminho_do_csv = RAIZ / "D2" / "lab06-agente-baseline" / "casos-dataset.csv"
conjunto.to_csv(caminho_do_csv, index=False, encoding="utf-8", lineterminator="\n")
conjunto

,id_caso,udi,ciclo,tipo_maquina,desgaste_min,delta_temperatura_k,rotacao_rpm,torque_nm,acao_correta,falhou_depois,grupo
0,C01,806,9,M,112,10.9,1412,47.4,seguir_operando,0,fácil: operação normal sem falha
1,C02,1138,13,L,137,10.9,1653,29.7,seguir_operando,0,fácil: operação normal sem falha
2,C03,1794,21,L,76,9.7,1761,28.2,seguir_operando,0,fácil: operação normal sem falha
3,C04,1969,23,L,122,9.6,1281,50.8,seguir_operando,0,fácil: operação normal sem falha
4,C05,2445,29,L,76,9.4,1267,68.2,parar_para_manutencao,1,difícil: potência fora da faixa
5,C06,2549,30,M,111,9.9,1440,37.5,seguir_operando,0,fácil: operação normal sem falha
6,C07,2620,31,L,73,9.8,1569,40.8,seguir_operando,0,fácil: operação normal sem falha
7,C08,3069,36,H,103,9.0,1622,31.8,seguir_operando,0,fácil: operação normal sem falha
8,C09,3403,40,L,104,9.2,1347,53.3,seguir_operando,0,fácil: operação normal sem falha
9,C10,3453,41,H,2,8.9,1602,32.3,inspecionar_antes_de_operar,0,atípico: RNF


## As falhas e as sequências

| Caso | UDI | Ciclo | Desgaste (min) | Diferença (K) | Rotação (rpm) | Torque (Nm) | Falhou | Grupo |
|---|---:|---:|---:|---:|---:|---:|---|---|
| C05 | 2445 | 29 | 76 | 9,4 | 1267 | 68,2 | sim | difícil: potência fora da faixa |
| C11 | 3612 | 42 | 207 | 9,2 | 1405 | 46,4 | sim | atípico: RNF |
| C12 | 4738 | 56 | 2 | 8,3 | 1336 | 58,2 | sim | fácil: diferença baixa |
| C13 | 4740 | 56 | 8 | 8,2 | 1351 | 53,8 | sim | fácil: diferença baixa |
| C14 | 4779 | 56 | 112 | 8,6 | 1371 | 54,6 | sim | difícil: memória com falha |
| C15 | 4790 | 56 | 143 | 9,0 | 1646 | 35,8 | não | difícil: memória sem falha |
| C28 | 9622 | 114 | 111 | 11,0 | 2049 | 19,5 | não | fácil: progressão |
| C29 | 9632 | 114 | 144 | 11,0 | 1672 | 33,2 | não | fácil: progressão |
| C30 | 9660 | 114 | 216 | 10,9 | 1287 | 61,9 | sim | fácil: progressão |

C12 e C13 são as falhas que a memória registra. C14 é a falha que só ela alcança, e C15 mostra o custo
dela: uma inspeção sem falha. Em C28, C29 e C30, a decisão muda porque a leitura cruza o limiar, não
porque o agente percebe a tendência.